# 🧮 Day 04a: Back-of-Envelope Estimation

---

## 🎯 What You'll Master Today
- QPS estimation framework
- Storage estimation
- Bandwidth estimation
- Memory/cache estimation

**Goal:** Estimate any system's requirements in 3-5 minutes.

---

```
╔═══════════════════════════════════════════════════════════════════╗
║  ESTIMATION POWER TABLE                                          ║
╠═══════════════════════════════════════════════════════════════════╣
║                                                                   ║
║  Power   │ Exact    │ Approx   │ Bytes                          ║
║  ────────│──────────│──────────│──────────                      ║
║  2^10    │ 1,024    │ 1 K      │ 1 KB                           ║
║  2^20    │ 1M       │ 1 M      │ 1 MB                           ║
║  2^30    │ 1B       │ 1 G      │ 1 GB                           ║
║  2^40    │ 1T       │ 1 T      │ 1 TB                           ║
║  2^50    │ 1Q       │ 1 P      │ 1 PB                           ║
║                                                                   ║
║  Time:                                                           ║
║  ────────│──────────│──────────                                  ║
║  1 day   │ 86,400 s │ ~100K s                                   ║
║  1 month │ 2.6M s   │ ~2.5M s                                   ║
║  1 year  │ 31.5M s  │ ~30M s                                    ║
║                                                                   ║
║  Data sizes:                                                     ║
║  char = 1 byte, int = 4 bytes, long = 8 bytes                  ║
║  UUID = 16 bytes, timestamp = 8 bytes                           ║
║  avg tweet = 200 bytes, avg image = 200 KB                      ║
║  avg video/min = 50 MB, avg user record = 500 bytes             ║
║                                                                   ║
╚═══════════════════════════════════════════════════════════════════╝
```

---

In [ ]:
# ============================================================
# 1. QPS Estimation Template
# ============================================================

def estimate_qps(dau, actions_per_user, seconds_in_day=86400, peak_factor=3):
    """
    Estimate QPS from daily active users.
    
    dau: daily active users
    actions_per_user: avg actions per user per day
    peak_factor: peak is typically 2-3x average
    """
    total_requests = dau * actions_per_user
    avg_qps = total_requests / seconds_in_day
    peak_qps = avg_qps * peak_factor
    
    print(f"DAU: {dau:,}")
    print(f"Actions/user/day: {actions_per_user}")
    print(f"Total daily requests: {total_requests:,}")
    print(f"Average QPS: {avg_qps:,.0f}")
    print(f"Peak QPS (×{peak_factor}): {peak_qps:,.0f}")
    return avg_qps, peak_qps

# Example: Twitter
print("=== Twitter Estimation ===")
print("\nREAD QPS (view tweets):")
estimate_qps(dau=300_000_000, actions_per_user=100)  # 100 tweet views/day
print("\nWRITE QPS (post tweets):")
estimate_qps(dau=300_000_000, actions_per_user=2)    # 2 tweets/day (active users)

In [ ]:
# ============================================================
# 2. Storage Estimation Template
# ============================================================

def estimate_storage(dau, items_per_user_per_day, item_size_bytes, years=5):
    """
    Estimate storage needs.
    """
    daily_items = dau * items_per_user_per_day
    daily_storage = daily_items * item_size_bytes
    yearly_storage = daily_storage * 365
    total_storage = yearly_storage * years
    
    def human_bytes(b):
        for unit in ['B', 'KB', 'MB', 'GB', 'TB', 'PB']:
            if b < 1024:
                return f"{b:.1f} {unit}"
            b /= 1024
        return f"{b:.1f} EB"
    
    print(f"Daily items: {daily_items:,}")
    print(f"Daily storage: {human_bytes(daily_storage)}")
    print(f"Yearly storage: {human_bytes(yearly_storage)}")
    print(f"{years}-year total: {human_bytes(total_storage)}")
    return total_storage

# Example: Twitter tweets
print("=== Twitter Tweet Storage ===")
estimate_storage(
    dau=300_000_000, 
    items_per_user_per_day=2, 
    item_size_bytes=300,  # tweet text + metadata
    years=5
)

# Example: Instagram images
print("\n=== Instagram Image Storage ===")
estimate_storage(
    dau=500_000_000,
    items_per_user_per_day=0.5,  # not everyone posts daily
    item_size_bytes=200_000,     # 200 KB avg image
    years=5
)

In [ ]:
# ============================================================
# 3. Bandwidth Estimation
# ============================================================

def estimate_bandwidth(daily_data_bytes, peak_factor=3):
    """
    Estimate bandwidth from daily data volume.
    """
    avg_bps = (daily_data_bytes * 8) / 86400  # bits per second
    peak_bps = avg_bps * peak_factor
    
    def human_bps(bps):
        for unit in ['bps', 'Kbps', 'Mbps', 'Gbps', 'Tbps']:
            if bps < 1000:
                return f"{bps:.1f} {unit}"
            bps /= 1000
        return f"{bps:.1f} Pbps"
    
    print(f"Average bandwidth: {human_bps(avg_bps)}")
    print(f"Peak bandwidth: {human_bps(peak_bps)}")

print("=== YouTube Video Streaming ===")
# 1B daily views × 5 min avg × 5 MB/min
daily_video_bytes = 1_000_000_000 * 5 * 5_000_000
estimate_bandwidth(daily_video_bytes)

print("\n=== WhatsApp Messages ===")
# 2B users × 50 messages × 500 bytes avg
daily_msg_bytes = 2_000_000_000 * 50 * 500
estimate_bandwidth(daily_msg_bytes)

In [ ]:
# ============================================================
# 4. Cache/Memory Estimation — 80/20 Rule
# ============================================================

print("""
CACHE ESTIMATION — The 80/20 Rule
══════════════════════════════════

Principle: 20% of data accounts for 80% of requests
→ Cache the hot 20% and you serve 80% from memory!

Formula:
  cache_size = daily_requests × avg_response_size × cache_ratio
  cache_ratio = 0.2 (cache 20% of daily data)

Example: Twitter timeline cache
""")

# Twitter: cache 20% of daily tweet reads
daily_read_requests = 300_000_000 * 100  # 300M DAU × 100 reads/day
avg_tweet_size = 300  # bytes
cache_ratio = 0.2

# But many reads hit the same tweets (viral content)
# Unique tweets accessed per day (assume 100M unique tweets)
unique_daily_tweets = 100_000_000
cache_size = unique_daily_tweets * avg_tweet_size * cache_ratio

print(f"  Unique daily tweets: {unique_daily_tweets:,}")
print(f"  Cache 20%: {unique_daily_tweets * cache_ratio:,.0f} tweets")
print(f"  Cache size: {cache_size / (1024**3):.1f} GB")
print(f"  → Fits in a single Redis instance (100GB+ possible)")

print("""
💡 Interview framework:
   1. Estimate total daily data
   2. Apply 80/20 rule (cache 20%)
   3. Check if it fits in memory (Redis can hold 100+ GB)
   4. If not, distribute across Redis cluster
""")

In [ ]:
# ============================================================
# 5. Complete Estimation Template
# ============================================================

print("""
╔═══════════════════════════════════════════════════════════════╗
║  ESTIMATION FRAMEWORK (Use in Every SD Interview)            ║
╠═══════════════════════════════════════════════════════════════╣
║                                                               ║
║  Step 1: USERS                                               ║
║    Monthly Active Users (MAU)                                ║
║    Daily Active Users (DAU) = MAU × 0.5 (typically)         ║
║                                                               ║
║  Step 2: QPS                                                 ║
║    Avg QPS = DAU × actions_per_user / 86400                 ║
║    Peak QPS = Avg × 3                                       ║
║    Read:Write ratio (most apps are 10:1 or 100:1)           ║
║                                                               ║
║  Step 3: STORAGE                                             ║
║    Daily new data = DAU × items × size_per_item             ║
║    5-year total = daily × 365 × 5                           ║
║    Add 30% overhead for indexes, metadata                   ║
║                                                               ║
║  Step 4: BANDWIDTH                                           ║
║    Avg = daily_data × 8 / 86400 (bits/sec)                  ║
║    Peak = Avg × 3                                           ║
║                                                               ║
║  Step 5: CACHE                                               ║
║    Cache 20% of daily data (80/20 rule)                     ║
║    Redis single node ≈ 100 GB                               ║
║                                                               ║
║  Key numbers to memorize:                                    ║
║    1 day = 86,400 sec ≈ 100K sec                           ║
║    1 year = 365 days ≈ 30M sec                              ║
║    char = 1B, int = 4B, UUID = 16B                          ║
║    tweet ≈ 300B, image ≈ 200KB, video/min ≈ 50MB           ║
║                                                               ║
╚═══════════════════════════════════════════════════════════════╝
""")

---

## 📝 Interview Questions

| # | Question | Key Points |
|---|----------|-----------|
| 1 | Estimate storage for Instagram | 500M DAU × 0.5 posts × 200KB × 365 × 5 = ~90 PB. Plus metadata, thumbnails |
| 2 | How many servers needed? | Calculate peak QPS ÷ single server capacity (e.g., 5000 QPS/server) |
| 3 | How much cache for Twitter? | 20% of daily unique tweets. 20M tweets × 300B ≈ 6GB. Fits one Redis |
| 4 | Estimate bandwidth for YouTube | 1B views × 5min × 5MB/min = 25PB/day. Peak ≈ 7 Tbps |
| 5 | What numbers should you memorize? | 1 day=100K sec, char=1B, image=200KB, DB=10K QPS, Redis=100K QPS |

---

## ┌──────────────────────────────────────────────────────────┐
## │              📌 KEY TAKEAWAYS                            │
## ├──────────────────────────────────────────────────────────┤
## │                                                          │
## │  • QPS = DAU × actions / 86400, peak = ×3               │
## │  • Storage = daily data × 365 × years + 30% overhead    │
## │  • Cache 20% of daily data (80/20 rule)                  │
## │  • Memorize: 1 day≈100K sec, image≈200KB, tweet≈300B    │
## │  • Always round UP for safety margin                     │
## │                                                          │
## └──────────────────────────────────────────────────────────┘

---

## ⏭️ Next Up: **Building Blocks — Load Balancer & CDN**